<a href="https://colab.research.google.com/github/mariakellyeduarda17-ops/UFV/blob/main/17004.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pysus pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.7/385.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import pandas as pd
from pysus.online_data import SIH

def extrair_e_salvar_csv(uf, ano, meses, cidades_alvo=None):
    resultados = []

    for mes in meses:
        print(f"Processando {uf} - {mes:02d}/{ano}...")
        try:
            # Download dos dados
            dataset = SIH.download(states=uf, years=ano, months=mes, groups='RD')
            df = dataset.to_dataframe()

            # Padronização e Filtro por ITU (N39.0)
            df['DIAG_PRINC'] = df['DIAG_PRINC'].str.strip()
            df_itu = df[df['DIAG_PRINC'] == 'N390'].copy()

            # Se houver lista de cidades específicas (ex: > 100k hab)
            if cidades_alvo:
                df_itu['MUNIC_RES'] = df_itu['MUNIC_RES'].astype(str).str.strip()
                df_itu = df_itu[df_itu['MUNIC_RES'].isin(cidades_alvo)]

            # Contagem
            if not df_itu.empty:
                contagem = df_itu.groupby(['MUNIC_RES']).size().reset_index(name='Internacoes')
                contagem['Mes'] = mes
                contagem['UF'] = uf
                resultados.append(contagem)

        except Exception as e:
            print(f"Erro no mês {mes} ({uf}): {e}")

    if resultados:
        df_final = pd.concat(resultados, ignore_index=True)
        filename = f'internacoes_itu_{uf.lower()}_{ano}.csv'
        # Salvando com separador ';' e encoding para Excel brasileiro
        df_final.to_csv(filename, index=False, sep=';', encoding='utf-8-sig')
        print(f"Arquivo {filename} gerado com sucesso!")
        return df_final
    else:
        print(f"Nenhum dado encontrado para {uf}.")
        return pd.DataFrame()

# --- CONFIGURAÇÃO ---
meses_2024 = [1, 2] # Exemplo com os primeiros meses
codigos_mg_100k = ['310620', '317020', '311860', '313670', '310670', '314330'] # etc.

# Execução
df_mg = extrair_e_salvar_csv('MG', 2024, meses_2024, cidades_alvo=codigos_mg_100k)
df_ac = extrair_e_salvar_csv('AC', 2024, meses_2024)

Processando MG - 01/2024...


RDMG2401.parquet: 100%|██████████| 429k/429k [00:25<00:00, 16.5kB/s]


Processando MG - 02/2024...


RDMG2402.parquet: 100%|██████████| 430k/430k [00:29<00:00, 14.5kB/s]


Arquivo internacoes_itu_mg_2024.csv gerado com sucesso!
Processando AC - 01/2024...


RDAC2401.parquet: 100%|██████████| 15.2k/15.2k [00:00<00:00, 20.0kB/s]


Processando AC - 02/2024...


RDAC2402.parquet: 100%|██████████| 15.2k/15.2k [00:00<00:00, 18.9kB/s]


Arquivo internacoes_itu_ac_2024.csv gerado com sucesso!


In [3]:
import pandas as pd
from pysus.online_data import SIH

def extrair_e_salvar_csv(uf, ano, meses, cidades_alvo=None):
    resultados = []

    for mes in meses:
        print(f"Processando {uf} - {mes:02d}/{ano}...")
        try:
            # Download dos dados
            dataset = SIH.download(states=uf, years=ano, months=mes, groups='RD')
            df = dataset.to_dataframe()

            # Padronização e Filtro por ITU (N39.0)
            df['DIAG_PRINC'] = df['DIAG_PRINC'].str.strip()
            df_itu = df[df['DIAG_PRINC'] == 'N390'].copy()

            # Se houver lista de cidades específicas (ex: > 100k hab)
            if cidades_alvo:
                df_itu['MUNIC_RES'] = df_itu['MUNIC_RES'].astype(str).str.strip()
                df_itu = df_itu[df_itu['MUNIC_RES'].isin(cidades_alvo)]

            # Contagem
            if not df_itu.empty:
                contagem = df_itu.groupby(['MUNIC_RES']).size().reset_index(name='Internacoes')
                contagem['Mes'] = mes
                contagem['UF'] = uf
                resultados.append(contagem)

        except Exception as e:
            print(f"Erro no mês {mes} ({uf}): {e}")

    if resultados:
        df_final = pd.concat(resultados, ignore_index=True)
        filename = f'internacoes_itu_{uf.lower()}_{ano}.csv'
        # Salvando com separador ';' e encoding para Excel brasileiro
        df_final.to_csv(filename, index=False, sep=';', encoding='utf-8-sig')
        print(f"Arquivo {filename} gerado com sucesso!")
        return df_final
    else:
        print(f"Nenhum dado encontrado para {uf}.")
        return pd.DataFrame()

# --- CONFIGURAÇÃO ---
meses_2024 = [1, 2] # Exemplo com os primeiros meses
codigos_mg_100k = ['310620', '317020', '311860', '313670', '310670', '314330'] # etc.

# Execução
df_mg = extrair_e_salvar_csv('MG', 2024, meses_2024, cidades_alvo=codigos_mg_100k)
df_ac = extrair_e_salvar_csv('AC', 2024, meses_2024)

Processando MG - 01/2024...


9548183it [00:00, 11638471999.31it/s]


Processando MG - 02/2024...


9466002it [00:00, 3358708235.56it/s] 


Arquivo internacoes_itu_mg_2024.csv gerado com sucesso!
Processando AC - 01/2024...


313213it [00:00, 496676952.27it/s]   


Processando AC - 02/2024...


316988it [00:00, 170476219.56it/s]   


Arquivo internacoes_itu_ac_2024.csv gerado com sucesso!
